# 数据获取与处理
## 1.1 数据导入

本notebook完成以下任务：

1. 从CSMAR下载的原始数据导入
2. 多张数据表合并
3. 变量构造
4. 样本筛选
5. 行业分类处理
6. Winsorize异常值处理

In [4]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import os

In [5]:
# 读取七张原始数据表（根据实际CSMAR列名映射）
print("=" * 60)
print("读取原始数据")
print("=" * 60)

# 1. 资产负债表 balance_sheet.csv (gbk编码)
balance = pd.read_csv("data/raw/balance_sheet.csv", encoding='gbk')
balance = balance.rename(columns={
    'Stkcd': 'stkcd',
    'Accper': 'year',
    '流动资产合计': 'current_assets',
    '固定资产净额': 'fixed_assets',
    '资产总计': 'totalassets',
    '流动负债合计': 'current_liabilities',
    '负债合计': 'total_liabilities'
})
balance['year'] = pd.to_datetime(balance['year']).dt.year
balance['stkcd'] = balance['stkcd'].astype(str)

# 2. 利润表 income_stmt.csv (gbk编码)
income = pd.read_csv("data/raw/income_stmt.csv", encoding='gbk')
income = income.rename(columns={
    'Stkcd': 'stkcd',
    'Accper': 'year',
    '净利润': 'net_profit'
})
income['year'] = pd.to_datetime(income['year']).dt.year
income['stkcd'] = income['stkcd'].astype(str)

# 3. 现金流量表 cashflow.csv (gbk编码)
cashflow = pd.read_csv("data/raw/cashflow.csv", encoding='gbk')
cashflow = cashflow.rename(columns={
    'Stkcd': 'stkcd',
    'Accper': 'year',
    '折旧摊销': 'da'
})
cashflow['year'] = pd.to_datetime(cashflow['year']).dt.year
cashflow['stkcd'] = cashflow['stkcd'].astype(str)

# 4. 股权性质 ownership.csv (gbk编码)
ownership = pd.read_csv("data/raw/ownership.csv", encoding='gbk')
ownership = ownership.rename(columns={
    'Symbol': 'stkcd',
    'EndDate': 'year',
    'EquityNatureID': 'controller_type_raw'
})
ownership['year'] = pd.to_datetime(ownership['year']).dt.year
ownership['stkcd'] = ownership['stkcd'].astype(str)

# ★★★ SOE映射规则（仅保留国有和民营）★★★
def map_to_soe(x):
    x = str(x).strip()
    if x == '1':
        return '国有企业'
    elif x == '2':
        return '民营企业'
    elif x == '1,2':
        return '民营企业'
    else:
        return 'DROP'

ownership['controller_type'] = ownership['controller_type_raw'].apply(map_to_soe)
ownership = ownership[ownership['controller_type'] != 'DROP'][['stkcd', 'year', 'controller_type']]
print(f"股权性质分布（筛选后）:")
print(ownership['controller_type'].value_counts())

# 5. 行业分类 industry.csv (gbk编码) — 使用IndustryCodeC的2位数代码
industry = pd.read_csv("data/raw/industry.csv", encoding='gbk')
industry = industry.rename(columns={
    'Symbol': 'stkcd',
    'EndDate': 'year',
    'IndustryCodeC': 'industry_code'
})
industry['year'] = pd.to_datetime(industry['year']).dt.year
industry['stkcd'] = industry['stkcd'].astype(str)

# 6. ST/PT标记 st_flag.csv
st_flag_raw = pd.read_csv("data/raw/st_flag.csv")
st_flag = st_flag_raw.rename(columns={'Stkcd': 'stkcd', 'ST': 'st_raw'})
st_flag['st_flag'] = st_flag['st_raw'].apply(lambda x: 'Normal' if x == 1 else 'ST')
st_flag = st_flag[['stkcd', 'st_flag']]
st_flag['stkcd'] = st_flag['stkcd'].astype(str)

# 7. M2数据 m2.csv (gbk编码)
m2 = pd.read_csv("data/raw/m2.csv", encoding='gbk')
m2 = m2.rename(columns={
    '统计期间': 'year',
    'M2': 'm2'
})
m2['year'] = pd.to_datetime(m2['year']).dt.year

print(f"\n资产负债表: {balance.shape}")
print(f"利润表: {income.shape}")
print(f"现金流量表: {cashflow.shape}")
print(f"股权性质: {ownership.shape}")
print(f"行业分类: {industry.shape}")
print(f"ST/PT标记: {st_flag.shape}")
print(f"M2数据: {m2.shape}")

读取原始数据
股权性质分布（筛选后）:
controller_type
民营企业    32068
国有企业    17846
Name: count, dtype: int64

资产负债表: (57879, 9)
利润表: (57879, 5)
现金流量表: (57966, 10)
股权性质: (49914, 3)
行业分类: (56650, 5)
ST/PT标记: (5717, 2)
M2数据: (16, 2)


In [6]:
# 查看各表的实际列名和数据（前几行）
print("\n【资产负债表列名】")
print(balance.columns.tolist())
print(balance.head(3).to_string())

print("\n【利润表列名】")
print(income.columns.tolist())
print(income.head(3).to_string())


【资产负债表列名】
['stkcd', 'ShortName', 'year', '报表类型', 'current_assets', 'fixed_assets', 'totalassets', 'current_liabilities', 'total_liabilities']
  stkcd ShortName  year 报表类型  current_assets  fixed_assets   totalassets  current_liabilities  total_liabilities
0     1      深发展A  2010    A             NaN  2.392293e+09  7.272071e+11                  NaN       6.940095e+11
1     1      深发展A  2011    A             NaN  3.524265e+09  1.258177e+12                  NaN       1.182796e+12
2     1      平安银行  2012    A             0.0  3.536443e+09  1.606537e+12                  0.0       1.521738e+12

【利润表列名】
['stkcd', 'ShortName', 'year', 'Typrep', 'net_profit']
  stkcd ShortName  year Typrep    net_profit
0     1      深发展A  2010      A  6.283816e+09
1     1      深发展A  2011      A  1.039049e+10
2     1      平安银行  2012      A  1.351078e+10


In [7]:
print("\n【现金流量表列名】")
print(cashflow.columns.tolist())
print(cashflow.head(3))

print("\n【股权性质列名】")
print(ownership.columns.tolist())
print(ownership.head(3))


【现金流量表列名】
['stkcd', 'ShortName', 'year', 'Typrep', 'D000103000', 'D000119000', 'D000120000', 'D000104000', 'D000105000', 'da']
  stkcd ShortName  year Typrep   D000103000  D000119000  D000120000  \
0     1      深发展A  2010      A  322211000.0         NaN         NaN   
1     1      深发展A  2011      A  457021000.0         NaN         NaN   
2     1      平安银行  2012      A  532397000.0         NaN         NaN   

    D000104000   D000105000            da  
0   52458000.0  117783000.0  4.924520e+08  
1  286890000.0  165949000.0  9.098600e+08  
2  501503000.0  222466000.0  1.256366e+09  

【股权性质列名】
['stkcd', 'year', 'controller_type']
   stkcd  year controller_type
16     2  2010            国有企业
17     2  2011            国有企业
18     2  2012            国有企业


In [8]:
print("\n【行业分类列名】")
print(industry.columns.tolist())
print(industry.head(3))

print("\n【ST/PT标记列名】")
print(st_flag.columns.tolist())
print(st_flag.head(3))

print("\n【M2数据列名】")
print(m2.columns.tolist())
print(m2.head(3))


【行业分类列名】
['stkcd', 'ShortName', 'year', 'IndustryName', 'industry_code']
  stkcd ShortName  year IndustryName industry_code
0     1      深发展A  2010       其他商业银行           J66
1     1      深发展A  2011       其他商业银行           J66
2     1      平安银行  2012       货币金融服务           J66

【ST/PT标记列名】
['stkcd', 'st_flag']
  stkcd st_flag
0     1  Normal
1     2  Normal
2     4      ST

【M2数据列名】
['year', 'm2']
   year        m2
0  2010  72585.18
1  2011  85159.09
2  2012  97414.88


## 1.3 数据合并

按 `stkcd` 和 `year` 合并七张数据表。

In [9]:
# 步骤1：合并三张财务报表（stkcd转字符串保留前导0）
balance['stkcd'] = balance['stkcd'].astype(str)
income['stkcd'] = income['stkcd'].astype(str)
cashflow['stkcd'] = cashflow['stkcd'].astype(str)

df = balance[['stkcd', 'year', 'totalassets', 'total_liabilities', 'fixed_assets', 
               'current_assets', 'current_liabilities']].copy()

df = df.merge(income[['stkcd', 'year', 'net_profit']], on=['stkcd', 'year'], how='inner')
df = df.merge(cashflow[['stkcd', 'year', 'da']], on=['stkcd', 'year'], how='inner')

# 步骤2：合并公司特征（inner join — 不在ownership表的公司直接剔除）
df = df.merge(ownership, on=['stkcd', 'year'], how='inner')

# 步骤3：合并行业分类
industry['stkcd'] = industry['stkcd'].astype(str)
df = df.merge(industry[['stkcd', 'year', 'industry_code']], 
              on=['stkcd', 'year'], how='left')

# ★★★ 补充2023-2025的行业分类（使用2022年的分类）★★★
industry_2022 = industry[industry['year'] == 2022][['stkcd', 'industry_code']].drop_duplicates('stkcd')
industry_2022 = industry_2022.rename(columns={'industry_code': 'industry_code_2022'})

# 对2023-2025缺失行业代码的行，用2022年数据填补
df = df.merge(industry_2022, on='stkcd', how='left')
df['industry_code'] = df['industry_code'].fillna(df['industry_code_2022'])
df = df.drop(columns=['industry_code_2022'])

print(f"行业代码填补完成！")
print(f"填补前 industry_code 缺失: {df['industry_code'].isna().sum()} 行")

# 步骤4：合并ST/PT标记
st_flag['stkcd'] = st_flag['stkcd'].astype(str)
df = df.merge(st_flag[['stkcd', 'st_flag']], on=['stkcd'], how='left')

# 步骤5：合并宏观数据
df = df.merge(m2, on='year', how='left')

print(f"合并后数据形状: {df.shape}")
print(f"\n股权性质分布（合并后）:")
print(df['controller_type'].value_counts())
print(f"\n行业代码缺失（填补后）: {df['industry_code'].isna().sum()}")

行业代码填补完成！
填补前 industry_code 缺失: 665 行
合并后数据形状: (49911, 13)

股权性质分布（合并后）:
controller_type
民营企业    32067
国有企业    17844
Name: count, dtype: int64

行业代码缺失（填补后）: 665


## 1.4 变量构造

按照作业要求定义所有变量：

In [10]:
# ========== 因变量 ==========
df['lev'] = df['total_liabilities'] / df['totalassets']

# ========== 核心解释变量 ==========
df['npr'] = df['net_profit'] / df['totalassets']

# ========== 控制变量 ==========
df['size'] = np.log(df['totalassets'])  # 对数总资产
df['tang'] = df['fixed_assets'] / df['totalassets']  # 有形资产率
df['ndts'] = df['da'] / df['totalassets']  # 非债务税盾

# 成长性（需要滞后一期）
df = df.sort_values(['stkcd', 'year'])
df['asset_lag'] = df.groupby('stkcd')['totalassets'].shift(1)
df['growth'] = (df['totalassets'] - df['asset_lag']) / df['asset_lag']

# 流动比率（选做）
df['liq'] = df['current_assets'] / df['current_liabilities']

# ========== SOE虚拟变量（inner join后controller_type不再有NaN）==========
df['soe'] = (df['controller_type'] == '国有企业').astype(int)

# ========== M2增长率 ==========
# m2按年份排序，用全局shift(1)取前一年值
m2_sorted = m2.sort_values('year').copy()
m2_sorted['m2_lag'] = m2_sorted['m2'].shift(1)
m2_sorted['m2_growth'] = (m2_sorted['m2'] - m2_sorted['m2_lag']) / m2_sorted['m2_lag'] * 100
# 只保留year和m2_growth
m2_growth_df = m2_sorted[['year', 'm2_growth']].dropna()

# 合并m2_growth
df = df.merge(m2_growth_df, on='year', how='left')

print("变量构造完成！")
print("\n关键变量预览：")
print(df[['stkcd', 'year', 'lev', 'npr', 'size', 'tang', 'growth', 'ndts', 'soe', 'm2_growth']].head(10))
print(f"\nSOE分布: 国有企业={df['soe'].sum()}, 民营企业={(df['soe']==0).sum()}")

变量构造完成！

关键变量预览：
  stkcd  year       lev       npr       size      tang    growth      ndts  \
0    10  2010  0.597420  0.018061  19.173047  0.143702       NaN  0.022772   
1    10  2011  0.606440  0.035149  19.288864  0.164117  0.122791  0.015500   
2    10  2012  0.602239  0.014986  19.295232  0.149445  0.006388  0.027487   
3    10  2013  0.358152  0.003708  20.750676  0.042490  3.286389  0.006169   
4    10  2014  0.314128 -0.093654  20.505381  0.059823 -0.217526  0.008703   
5    10  2015  0.579577  0.003869  22.364625  0.011085  5.418882  0.001775   
6    10  2016  0.544721  0.008022  22.302922  0.014374 -0.059838  0.003446   
7    10  2017  0.620116 -0.338150  21.856329  0.020223 -0.360196  0.006298   
8    10  2018  0.819505 -0.221953  21.909058  0.021700  0.054144  0.002749   
9    10  2019  0.824373  0.029389  22.179939  0.010274  0.311120  0.003726   

   soe  m2_growth  
0    0        NaN  
1    0  17.322971  
2    0  14.391640  
3    0  13.588910  
4    0  11.011934  
5   

In [11]:
# =============================================================================
# ★★★ 样本筛选 ★★★
# 注意：剔除ST/PT公司的步骤放在最后统一执行
# =============================================================================

# 记录筛选过程（剔除前先记录初始状态）
screening_log = []

# 初始样本
initial_n = len(df)
initial_firms = df['stkcd'].nunique()
screening_log.append(['初始样本', 0, initial_n, initial_firms])
print(f"初始样本: {initial_n} 观测, {initial_firms} 家公司")

# ========== 步骤1：标记ST/PT公司（暂不剔除）==========
# 识别曾被ST/PT的公司
st_firms_all = df[df['st_flag'].isin(['ST', 'PT'])]['stkcd'].unique()
df['ever_st'] = df['stkcd'].isin(st_firms_all).astype(int)

print(f"\n【ST/PT标记统计】")
print(f"曾被ST/PT的公司数: {len(st_firms_all)}")
print(f"这些公司涉及观测数: {df['ever_st'].sum()}")
print(f"标记变量 'ever_st' 已添加，稍后统一剔除")

# ========== 步骤2：剔除金融保险行业（证监会行业代码 J 开头）==========
df = df[~df['industry_code'].astype(str).str.startswith('J', na=False)]
n_removed = initial_n - len(df)
n_firms = df['stkcd'].nunique()
screening_log.append(['剔除金融保险', n_removed, len(df), n_firms])
print(f"\n剔除金融保险: -{n_removed}, 剩余 {len(df)} 观测, {n_firms} 家公司")

# ========== 步骤3：剔除资不抵债（lev > 1）==========
df = df[df['lev'] <= 1]
n_removed = initial_n - len(df)
n_firms = df['stkcd'].nunique()
screening_log.append(['剔除Lev>1', n_removed, len(df), n_firms])
print(f"剔除资不抵债: -{n_removed}, 剩余 {len(df)} 观测, {n_firms} 家公司")

# ========== 步骤4：剔除关键变量缺失==========
key_vars = ['lev', 'npr', 'size', 'tang', 'growth', 'ndts', 'soe']
df_before_na = len(df)
df = df.dropna(subset=key_vars)
n_removed = df_before_na - len(df)
n_firms = df['stkcd'].nunique()
screening_log.append(['剔除缺失值', n_removed, len(df), n_firms])
print(f"剔除缺失值: -{n_removed}, 剩余 {len(df)} 观测, {n_firms} 家公司")

# ========== ★★★ 最终剔除：ST/PT公司（保守处理）==========
n_before_st = len(df)
st_firms_final = df[df['ever_st'] == 1]['stkcd'].unique()
df = df[df['ever_st'] == 0]
n_removed = n_before_st - len(df)
n_firms = df['stkcd'].nunique()
screening_log.append(['★最终剔除ST/PT★', n_removed, len(df), n_firms])
print(f"\n★最终剔除ST/PT公司: -{n_removed}, 剩余 {len(df)} 观测, {n_firms} 家公司")
print(f"   （剔除原因：只要曾被ST过，全部年度均剔除）")

# 删除辅助标记变量
df = df.drop(columns=['ever_st'])

初始样本: 49911 观测, 5240 家公司

【ST/PT标记统计】
曾被ST/PT的公司数: 757
这些公司涉及观测数: 9630
标记变量 'ever_st' 已添加，稍后统一剔除

剔除金融保险: -867, 剩余 49044 观测, 5175 家公司
剔除资不抵债: -1222, 剩余 48689 观测, 5174 家公司
剔除缺失值: -5129, 剩余 43560 观测, 5052 家公司

★最终剔除ST/PT公司: -8480, 剩余 35080 观测, 4303 家公司
   （剔除原因：只要曾被ST过，全部年度均剔除）


In [12]:
# 呈现样本筛选流程表
screening_df = pd.DataFrame(screening_log, 
                            columns=['筛选步骤', '剔除观测数', '剩余观测数', '剩余公司数'])
print("\n" + "=" * 70)
print("样本筛选流程表")
print("=" * 70)
print(screening_df.to_string(index=False))

# 保存筛选表
screening_df.to_csv('output/screening_table.csv', index=False, encoding='utf-8-sig')


样本筛选流程表
       筛选步骤  剔除观测数  剩余观测数  剩余公司数
       初始样本      0  49911   5240
     剔除金融保险    867  49044   5175
    剔除Lev>1   1222  48689   5174
      剔除缺失值   5129  43560   5052
★最终剔除ST/PT★   8480  35080   4303


## 1.6 行业分类处理

制造业（C开头）用2位行业代码，其他行业用1位代码；样本量<30的制造业子行业合并为"其他制造业"。

In [13]:
# 行业分类处理函数
# 制造业（C开头）：使用3位数行业代码（C39、C13、C26等）
# 其他行业：使用1位数行业代码（K70→K，I65→I等）
# 制造业2位数子行业样本量<30 → 合并为"其他制造业"
def classify_industry(code):
    if pd.isna(code) or str(code).strip() == '':
        return 'Unknown'
    code_str = str(code).strip()
    if code_str.startswith('C'):
        return code_str[:3]  # 制造业取前3位，如C39、C13、C26
    else:
        return code_str[0]   # 其他行业取前1位，如K70→K，I65→I

df['ind_code'] = df['industry_code'].apply(classify_industry)

# 统计各行业样本量
ind_counts = df['ind_code'].value_counts()
print("各行业样本量分布：")
print(ind_counts.sort_index())

# 样本量<30的制造业子行业合并为"其他制造业"
small_mfg = ind_counts[(ind_counts < 30) & (ind_counts.index.str.startswith('C'))].index.tolist()
if len(small_mfg) > 0:
    df.loc[df['ind_code'].isin(small_mfg), 'ind_code'] = 'C_OTHER'
    print(f"\n已合并以下制造业子行业: {small_mfg}")

print("\n处理后行业分布：")
print(df['ind_code'].value_counts().sort_index())

各行业样本量分布：
ind_code
A           391
B           676
C13         455
C14         440
C15         394
C17         415
C18         347
C19          69
C20          62
C21         191
C22         307
C23          78
C24         145
C25         119
C26        2280
C27        2309
C28         218
C29         859
C30         909
C31         286
C32         632
C33         716
C34        1391
C35        2262
C36        1238
C37         523
C38        2282
C39        3470
C40         527
C41         147
C42          51
C43           1
D          1068
E           802
F          1634
G          1124
H           101
I          2619
K           974
L           436
M           519
N           540
O            15
P            33
Q            67
R           482
S           161
Unknown     315
Name: count, dtype: int64

已合并以下制造业子行业: ['C43']

处理后行业分布：
ind_code
A           391
B           676
C13         455
C14         440
C15         394
C17         415
C18         347
C19          69
C20          62
C2

## 1.7 Winsorize异常值处理

对连续变量在截面层面（每年）进行双侧1% Winsorize。

In [14]:
from scipy.stats.mstats import winsorize

# 需要Winsorize的变量
winsor_vars = ['lev', 'npr', 'tang', 'growth', 'ndts', 'liq']

# 保存原始数据用于对比
df_raw = df.copy()

# 按年分别进行Winsorize
df = df.copy()
for var in winsor_vars:
    if var in df.columns:
        df[var] = df.groupby('year')[var].transform(
            lambda x: winsorize(x, limits=[0.01, 0.01])
        )
        print(f"{var} Winsorize完成!")

print("\nWinsorize处理完成！")

lev Winsorize完成!
npr Winsorize完成!
tang Winsorize完成!
growth Winsorize完成!
ndts Winsorize完成!
liq Winsorize完成!

Winsorize处理完成！


In [15]:
# Winsorize前后对比统计
print("=" * 70)
print("Winsorize前后对比")
print("=" * 70)

compare_vars = ['lev', 'npr', 'growth']

for var in compare_vars:
    print(f"\n【{var}】")
    print(f"  原始数据 - Mean: {df_raw[var].mean():.4f}, Std: {df_raw[var].std():.4f}, Min: {df_raw[var].min():.4f}, Max: {df_raw[var].max():.4f}")
    print(f"  Winsorize - Mean: {df[var].mean():.4f}, Std: {df[var].std():.4f}, Min: {df[var].min():.4f}, Max: {df[var].max():.4f}")

Winsorize前后对比

【lev】
  原始数据 - Mean: 0.4052, Std: 0.1963, Min: 0.0075, Max: 0.9974
  Winsorize - Mean: 0.4049, Std: 0.1950, Min: 0.0320, Max: 0.8958

【npr】
  原始数据 - Mean: 0.0368, Std: 0.0660, Min: -1.2401, Max: 0.7586
  Winsorize - Mean: 0.0372, Std: 0.0576, Min: -0.3385, Max: 0.2227

【growth】
  原始数据 - Mean: 0.3076, Std: 25.2225, Min: -0.9312, Max: 4719.6119
  Winsorize - Mean: 0.1416, Std: 0.2784, Min: -0.3184, Max: 3.5559


## 1.8 保存清洗后数据

In [16]:
# ★★★ 剔除行业代码为Unknown的行（原始数据中不存在的公司）★★★
n_before_unknown = len(df)
df = df[df['ind_code'] != 'Unknown']
n_removed = n_before_unknown - len(df)
print(f"剔除 ind_code=Unknown: -{n_removed} 行, 剩余 {len(df)} 行")

# 保存清洗后的面板数据
df.to_csv("data/clean/panel_data.csv", index=False)

# 同时保存为Stata格式（.dta）供Stata回归使用
# 删除controller_type列（避免Stata导出错误）+ st_flag列
df_export = df.drop(columns=['controller_type', 'st_flag'], errors='ignore')
df_export.to_stata("data/clean/panel_data.dta", write_index=False)

print("=" * 60)
print("数据处理完成！")
print("=" * 60)
print(f"最终样本: {len(df):,} 观测, {df['stkcd'].nunique():,} 家公司")
print(f"年份范围: {df['year'].min()} - {df['year'].max()}")
print(f"SOE: 国有企业={df['soe'].sum():,}, 民营企业={(df['soe']==0).sum():,}")
print("\n保存文件：")
print("  - data/clean/panel_data.csv")
print("  - data/clean/panel_data.dta")
print("  - output/screening_table.csv")

剔除 ind_code=Unknown: -315 行, 剩余 34765 行
数据处理完成！
最终样本: 34,765 观测, 4,046 家公司
年份范围: 2011 - 2025
SOE: 国有企业=12,202, 民营企业=22,563

保存文件：
  - data/clean/panel_data.csv
  - data/clean/panel_data.dta
  - output/screening_table.csv


In [17]:
# 最终数据预览
print("最终数据变量列表：")
print(df.columns.tolist())
print("\n数据前10行：")
print(df.head(10).to_string())

最终数据变量列表：
['stkcd', 'year', 'totalassets', 'total_liabilities', 'fixed_assets', 'current_assets', 'current_liabilities', 'net_profit', 'da', 'controller_type', 'industry_code', 'st_flag', 'm2', 'lev', 'npr', 'size', 'tang', 'ndts', 'asset_lag', 'growth', 'liq', 'soe', 'm2_growth', 'ind_code']

数据前10行：
   stkcd  year   totalassets  total_liabilities  fixed_assets  current_assets  current_liabilities   net_profit           da controller_type industry_code st_flag         m2       lev       npr       size      tang      ndts     asset_lag    growth       liq  soe  m2_growth ind_code
16    11  2011  3.499608e+09       2.368502e+09  6.501154e+07    2.827201e+09         2.239468e+09  257461077.5  35200034.90            国有企业           K70  Normal   85159.09  0.676791  0.073569  21.975917  0.018577  0.010058  2.913281e+09  0.201260  1.262443    1  17.322971        K
17    11  2012  3.950706e+09       2.446687e+09  7.882117e+07    3.190070e+09         2.278634e+09  374822152.4  40838063.78     